# BYV-AI Training v0.1.2 — Qwen3-1.7B + QLoRA (Colab T4 / FP16 dtype fix)

Notebook Colab untuk membangun adapter BYV-AI pertama, dengan pemuatan dtype eksplisit untuk menghindari parameter BF16 pada runtime T4. Preset awal ini disesuaikan untuk hasil runtime pengguna: NVIDIA Tesla T4, sekitar 14.56 GiB VRAM, compute capability 7.5, dengan FP16 (bukan BF16). Model dasar: **`Qwen/Qwen3-1.7B`**. Target perilaku: **thinking-capable, text-only, plain text, Markdown, source-code generation, dan DSML-as-text** — tanpa native tool calling.

> **Status:** eksperimen awal. Notebook ini sengaja tidak memulai training otomatis. Dataset belum tersedia; notebook membuat contoh smoke-test dan template, lalu menunggu dataset yang sudah dikurasi sebelum training sungguhan.

## Yang dikerjakan notebook ini

1. Cek GPU/VRAM yang benar-benar diberikan Colab.
2. Hubungkan Google Drive dan buat struktur folder BYV-AI.
3. Siapkan contoh JSONL starter dan template dataset.
4. Validasi format percakapan dan DSML.
5. Uji baseline model sebelum fine-tuning (jika GPU tersedia).
6. Jalankan QLoRA 4-bit NF4 setelah dataset siap.
7. Simpan adapter, tokenizer, log, dan manifest eksperimen ke Drive.
8. Bandingkan output sebelum/sesudah training pada prompt plain text, Markdown, coding, dan DSML.

**Catatan penting tentang thinking:** Qwen3 mendukung `enable_thinking=True` pada chat template. Kita melatih jawaban assistant (final content), bukan mengumpulkan atau mengajarkan private chain-of-thought mentah. Reasoning tetap menjadi perilaku model saat inferensi; jawaban akhir boleh berisi ringkasan penjelasan yang berguna.

## 0. Sebelum menjalankan

Di Colab pilih **Runtime → Change runtime type → T4 GPU** (atau GPU lain jika tersedia). GPU Colab Free tidak dijamin selalu tersedia. Notebook akan tetap menyiapkan folder dan dataset bila GPU tidak tersedia, tetapi cell training akan berhenti dengan pesan yang jelas.

Urutan yang disarankan: jalankan cell dari atas ke bawah. Setelah install library, bila Colab meminta restart runtime, lakukan restart lalu lanjutkan lagi dari cell awal.

Folder Drive yang dibuat:

```text
MyDrive/BYV-AI/
├── data/
│   ├── train.jsonl                 # dataset utama yang lu isi
│   ├── eval.jsonl                  # opsional; kalau tidak ada, train di-split
│   ├── dataset_template.jsonl
│   └── starter_smoke_test.jsonl
├── outputs/
│   └── qwen3-1.7b-qlora-v0.1/
└── experiments/
```

## 1. Konfigurasi eksperimen

Ubah nilai di cell ini sesuai kondisi setelah GPU dicek. Default dibuat konservatif untuk eksperimen awal.

In [1]:
from pathlib import Path
import os, json, random, textwrap, platform, sys

MODEL_ID = "Qwen/Qwen3-1.7B"
RUN_NAME = "byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2"
SEED = 42

# Preset awal untuk Tesla T4 16 GB class: 4-bit NF4, FP16 compute, sequence 1024, micro-batch 1.
# Jangan naikkan context/batch sampai training smoke test berhasil dan VRAM peak diperiksa.

# Dataset dan training
USE_STARTER_SMOKE_TEST = False  # True hanya untuk smoke-test pipeline, BUKAN training model rilis
ALLOW_SMALL_DATASET_FOR_SMOKE_TEST = False
MIN_TRAIN_EXAMPLES = 50
MAX_LENGTH = 1024              # Mulai konservatif; naikkan setelah VRAM terukur
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

# Inferensi evaluasi
EVAL_MAX_NEW_TOKENS = 1200

random.seed(SEED)
print("Model:", MODEL_ID)
print("Run:", RUN_NAME)
print("Python:", sys.version.split()[0])

Model: Qwen/Qwen3-1.7B
Run: byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2
Python: 3.13.16


## 2. Install dependencies

Jangan install/upgrade PyTorch secara terpisah di cell ini; Colab biasanya sudah menyediakan build PyTorch yang cocok dengan CUDA runtime-nya. Kita pasang stack Hugging Face yang diperlukan dan catat versinya setelah import.

In [16]:
%pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub

Jika muncul permintaan **Restart session**, restart runtime, lalu jalankan kembali cell dari awal. Cell install boleh dilewati setelah library berhasil terpasang.

## 3. Cek GPU dan VRAM

Cell ini tidak mengasumsikan T4/L4/A100. Informasi aktual dari runtime yang menentukan dtype dan apakah QLoRA bisa dijalankan.

In [3]:
import torch

GPU_AVAILABLE = torch.cuda.is_available()
GPU_NAME = None
GPU_VRAM_GB = 0.0
GPU_COMPUTE_CAPABILITY = None
BF16_SUPPORTED = False  # Native hardware BF16 support, not emulation support.
BF16_EMULATION_REPORTED = False

print("CUDA available:", GPU_AVAILABLE)
if GPU_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    GPU_VRAM_GB = round(props.total_memory / (1024 ** 3), 2)
    capability = torch.cuda.get_device_capability(0)
    GPU_COMPUTE_CAPABILITY = f"{capability[0]}.{capability[1]}"

    # Native NVIDIA BF16 Tensor Core support starts at compute capability 8.0.
    # torch.cuda.is_bf16_supported() defaults to including emulation checks, so
    # it can report True on older devices such as Tesla T4 (CC 7.5).
    BF16_SUPPORTED = capability[0] >= 8
    try:
        BF16_EMULATION_REPORTED = torch.cuda.is_bf16_supported()
    except Exception:
        BF16_EMULATION_REPORTED = False

    print("GPU:", GPU_NAME)
    print("VRAM:", GPU_VRAM_GB, "GB")
    print("Compute capability:", GPU_COMPUTE_CAPABILITY)
    print("BF16 reported by PyTorch (may include emulation):", BF16_EMULATION_REPORTED)
    print("Native BF16 hardware support:", BF16_SUPPORTED)
    print("CUDA:", torch.version.cuda)

    COMPUTE_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16
    print("Selected compute dtype:", COMPUTE_DTYPE)

    if capability[0] < 8:
        print("NOTE: Using FP16. This is the appropriate native Tensor Core precision for NVIDIA T4/Turing GPUs.")
else:
    COMPUTE_DTYPE = torch.float32
    print("GPU belum tersedia. Setup data tetap bisa dilanjutkan; training QLoRA GPU akan dilewati.")

print("Compute dtype:", COMPUTE_DTYPE)


CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB
Compute capability: 7.5
BF16 reported by PyTorch (may include emulation): True
Native BF16 hardware support: False
CUDA: 13.0
Selected compute dtype: torch.float16
NOTE: Using FP16. This is the appropriate native Tensor Core precision for NVIDIA T4/Turing GPUs.
Compute dtype: torch.float16


### Catatan dtype untuk GPU saat ini: Tesla T4\n\nHasil runtime lu menunjukkan Tesla T4 dengan compute capability 7.5. Untuk GPU ini notebook memilih **FP16**. Output `torch.cuda.is_bf16_supported()` dapat bernilai True karena pemeriksaan PyTorch secara default mengizinkan emulasi; itu bukan bukti bahwa T4 memiliki native BF16 Tensor Core. Karena itu, training config akan memakai `fp16=True` dan `bf16=False`.\n

## 4. Hubungkan Google Drive

Drive digunakan untuk menyimpan dataset dan checkpoint agar tidak hilang ketika runtime Colab terputus.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/BYV-AI')
DATA_DIR = DRIVE_ROOT / 'data'
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / RUN_NAME
EXPERIMENT_DIR = DRIVE_ROOT / 'experiments' / RUN_NAME

for folder in [DATA_DIR, OUTPUT_DIR, EXPERIMENT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.jsonl'
EVAL_PATH = DATA_DIR / 'eval.jsonl'
TEMPLATE_PATH = DATA_DIR / 'dataset_template.jsonl'
SMOKE_PATH = DATA_DIR / 'starter_smoke_test.jsonl'

print("Drive root:", DRIVE_ROOT)
print("Train file:", TRAIN_PATH)
print("Eval file:", EVAL_PATH)
print("Output:", OUTPUT_DIR)

Mounted at /content/drive
Drive root: /content/drive/MyDrive/BYV-AI
Train file: /content/drive/MyDrive/BYV-AI/data/train.jsonl
Eval file: /content/drive/MyDrive/BYV-AI/data/eval.jsonl
Output: /content/drive/MyDrive/BYV-AI/outputs/byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2


## 5. Siapkan system prompt dan contoh dataset starter

Starter dataset di bawah hanya contoh format dan **smoke-test**. Isinya terlalu kecil untuk menghasilkan model yang bagus. Untuk training yang serius, kembangkan menjadi ratusan contoh berkualitas, seimbang, dan ditinjau manusia.

Dataset menyimpan seluruh percakapan sebagai `messages` dengan role `system`, `user`, dan `assistant`. DSML adalah teks di `assistant.content`, bukan metadata `tool_calls`.

In [5]:
BASE_SYSTEM = """You are BYV-AI, a helpful reasoning language assistant.
Follow the user's instructions carefully. Think through tasks before answering, but do not expose private chain-of-thought. Provide useful concise explanations when appropriate.
Use plain text by default. Use Markdown when it improves readability. Generate complete source code when requested, preserving syntax and indentation. You can work with HTML, CSS, JavaScript, TypeScript, Python, SQL, and other programming languages.
Generate DSML only when an available tool is required and the current request needs it. DSML is ordinary assistant text, not native tool calling. Never invent tools, parameters, or execution results. Never claim an action succeeded before the gateway returns an actual result.
"""

DEMO_TOOL_PROMPT = """You have access to the tools listed below. These are the only tools available to you for this request.
- Bash(command*: string): Run an allowed shell command in the configured workspace.
- ReadFile(path*: string): Read a text file from the configured workspace.
- ListFiles(path*: string): List files in a directory.
- GitStatus(): Return repository status.
- SearchFiles(pattern*: string, options: object): Search workspace files using the supplied filter options.
MANDATORY RULES:
1. If the task requires an action one of these tools can perform, use the appropriate tool.
2. Use gateway DSML as assistant text. Do not use native tool calls or a legacy JSON wrapper.
3. Emit exactly one outer calls block with one invoke block per requested tool.
4. For string arguments use string=\"true\" and raw string content. For non-string JSON values use string=\"false\" and valid JSON.
5. Use exact tool and parameter names. Do not invent tools or parameters.
6. Do not put prose or Markdown inside a calls block; emit nothing outside the calls block when requesting tools.
7. If no tool is needed, answer with plain text or Markdown.
8. After actual tool results are returned, answer the user using plain text or Markdown. Do not emit another DSML block unless another tool is needed.
9. Never fabricate tool results.
DSML example:
<｜DSML｜ calls>
<｜DSML｜ invoke name=\"Bash\">
<｜DSML｜ parameter name=\"command\" string=\"true\">pwd</｜DSML｜ parameter>
</｜DSML｜ invoke>
</｜DSML｜ calls>
"""

def make_record(sample_id, category, user_text, assistant_text, tools=False, extra_messages=None):
    system = BASE_SYSTEM + ("\n\n" + DEMO_TOOL_PROMPT if tools else "")
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user_text}]
    if extra_messages:
        messages.extend(extra_messages)
    messages.append({"role": "assistant", "content": assistant_text})
    return {"id": sample_id, "category": category, "messages": messages}

STARTER_RECORDS = [
    make_record("plain-001", "plain_text", "Siapa kamu? Jawab singkat dalam bahasa Indonesia.",
                "Saya BYV-AI, asisten AI yang membantu analisis, penalaran, pemrograman, dan penyusunan solusi."),
    make_record("plain-002", "plain_text", "Jelaskan perbedaan HTML dan CSS.",
                "HTML menentukan struktur dan konten halaman web, sedangkan CSS mengatur tampilan seperti warna, jarak, ukuran, dan tata letak." , tools=True),
    make_record("markdown-001", "markdown", "Buat tabel singkat perbedaan frontend dan backend.",
                "| Aspek | Frontend | Backend |\n|---|---|---|\n| Peran | Antarmuka pengguna | Logika dan layanan server |\n| Contoh | HTML, CSS, JavaScript | Python, PHP, Go |\n| Berjalan di | Umumnya browser | Umumnya server |"),
    make_record("html-001", "code_html", "Buat halaman HTML sederhana bertajuk BYV-AI.",
                "```html\n<!DOCTYPE html>\n<html lang=\"id\">\n<head>\n  <meta charset=\"UTF-8\">\n  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">\n  <title>BYV-AI</title>\n</head>\n<body>\n  <main>\n    <h1>Selamat datang di BYV-AI</h1>\n    <p>Asisten AI berbasis reasoning dan teks.</p>\n  </main>\n</body>\n</html>\n```"),
    make_record("css-001", "code_css", "Buat CSS untuk card responsif dengan sudut membulat.",
                "```css\n.card {\n  width: min(100% - 2rem, 32rem);\n  margin-inline: auto;\n  padding: 1.5rem;\n  border-radius: 1rem;\n  background: #ffffff;\n  box-shadow: 0 8px 24px rgb(0 0 0 / 0.08);\n}\n```"),
    make_record("javascript-001", "code_javascript", "Tulis fungsi JavaScript untuk menjumlahkan dua angka.",
                "```javascript\nfunction add(a, b) {\n  return a + b;\n}\n\nconsole.log(add(2, 3)); // 5\n```"),
    make_record("python-001", "code_python", "Buat fungsi Python untuk membaca file JSON dengan penanganan error.",
                "```python\nimport json\nfrom pathlib import Path\n\ndef read_json(path):\n    try:\n        return json.loads(Path(path).read_text(encoding=\"utf-8\"))\n    except (OSError, json.JSONDecodeError) as exc:\n        raise ValueError(f\"Gagal membaca JSON: {exc}\") from exc\n```"),
    make_record("sql-001", "code_sql", "Tulis SQL untuk mengambil 10 pengguna terbaru dari tabel users.",
                "```sql\nSELECT id, name, created_at\nFROM users\nORDER BY created_at DESC\nLIMIT 10;\n```"),
    make_record("dsml-001", "dsml_single", "Tolong tampilkan direktori kerja saat ini.",
                "<｜DSML｜ calls>\n<｜DSML｜ invoke name=\"Bash\">\n<｜DSML｜ parameter name=\"command\" string=\"true\">pwd</｜DSML｜ parameter>\n</｜DSML｜ invoke>\n</｜DSML｜ calls>", tools=True),
    make_record("dsml-002", "dsml_single", "Baca file README.md di workspace.",
                "<｜DSML｜ calls>\n<｜DSML｜ invoke name=\"ReadFile\">\n<｜DSML｜ parameter name=\"path\" string=\"true\">README.md</｜DSML｜ parameter>\n</｜DSML｜ invoke>\n</｜DSML｜ calls>", tools=True),
    make_record("dsml-003", "dsml_multiple", "Daftarkan file di folder project dan periksa status Git. Kedua hal ini tidak saling bergantung.",
                "<｜DSML｜ calls>\n<｜DSML｜ invoke name=\"ListFiles\">\n<｜DSML｜ parameter name=\"path\" string=\"true\">project</｜DSML｜ parameter>\n</｜DSML｜ invoke>\n<｜DSML｜ invoke name=\"GitStatus\">\n</｜DSML｜ invoke>\n</｜DSML｜ calls>", tools=True),
    make_record("no-tool-with-tools-001", "plain_text_with_tools", "Dengan tools yang tersedia, jelaskan apa fungsi file README.",
                "README adalah dokumen pengantar proyek. Biasanya berisi tujuan, cara instalasi, konfigurasi, contoh penggunaan, dan informasi penting bagi kontributor.", tools=True),
    make_record("clarify-001", "clarification", "Bacakan file yang tadi saya maksud.",
                "File yang mana yang ingin lu baca? Sebutkan path atau nama filenya terlebih dahulu.", tools=True),
    make_record("dsml-result-001", "tool_result_to_markdown", "Gateway tool result (ReadFile, README.md):\n# BYV-AI\nProject ini adalah asisten text-generation yang mendukung DSML sebagai teks.\n\nTolong rangkum isi file tersebut.",
                "## Ringkasan README\n\n**BYV-AI** adalah proyek asisten AI yang menghasilkan teks biasa atau DSML sebagai teks.\n\nPoin utama:\n- Model tidak memakai native tool calling.\n- DSML diproses oleh gateway terpisah.\n- Output perlu divalidasi sebelum tindakan dijalankan.", tools=True),
    make_record("sequence-001", "multi_turn_sequence", "Temukan file konfigurasi di root repository, lalu baca isinya.",
                "<｜DSML｜ calls>\n<｜DSML｜ invoke name=\"ListFiles\">\n<｜DSML｜ parameter name=\"path\" string=\"true\">.</｜DSML｜ parameter>\n</｜DSML｜ invoke>\n</｜DSML｜ calls>", tools=True,
                extra_messages=[
                    {"role": "user", "content": "Gateway tool result (ListFiles): ditemukan config.yaml dan README.md. Lanjutkan membaca config.yaml."},
                    {"role": "assistant", "content": "<｜DSML｜ calls>\n<｜DSML｜ invoke name=\"ReadFile\">\n<｜DSML｜ parameter name=\"path\" string=\"true\">config.yaml</｜DSML｜ parameter>\n</｜DSML｜ invoke>\n</｜DSML｜ calls>"},
                    {"role": "user", "content": "Gateway tool result (ReadFile, config.yaml):\nmodel: qwen3\ncontext: 8192\nTolong rangkum konfigurasi ini."}
                ]),
    make_record("dsml-object-001", "dsml_parameters", "Cari file yang berisi kata config dengan opsi recursive bernilai true.",
                "<｜DSML｜ calls>\n<｜DSML｜ invoke name=\"SearchFiles\">\n<｜DSML｜ parameter name=\"pattern\" string=\"true\">config</｜DSML｜ parameter>\n<｜DSML｜ parameter name=\"options\" string=\"false\">{\"recursive\": true}</｜DSML｜ parameter>\n</｜DSML｜ invoke>\n</｜DSML｜ calls>", tools=True),
]

# Starter file for smoke tests and a small template that users can copy/edit.
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

if not SMOKE_PATH.exists():
    write_jsonl(SMOKE_PATH, STARTER_RECORDS)
if not TEMPLATE_PATH.exists():
    write_jsonl(TEMPLATE_PATH, [
        make_record("replace-me-001", "plain_text", "Tulis instruksi pengguna di sini.",
                    "Tulis jawaban assistant yang diharapkan di sini.")
    ])

print("Starter smoke-test:", SMOKE_PATH)
print("Jumlah contoh starter:", len(STARTER_RECORDS))
print("Template dataset:", TEMPLATE_PATH)
print("PENTING: starter dataset tidak digunakan untuk training kecuali USE_STARTER_SMOKE_TEST=True.")

Starter smoke-test: /content/drive/MyDrive/BYV-AI/data/starter_smoke_test.jsonl
Jumlah contoh starter: 16
Template dataset: /content/drive/MyDrive/BYV-AI/data/dataset_template.jsonl
PENTING: starter dataset tidak digunakan untuk training kecuali USE_STARTER_SMOKE_TEST=True.



### Opsional: upload file JSONL dari komputer

Kalau dataset sudah dibuat di komputer, jalankan cell berikut lalu pilih `train.jsonl` dan (opsional) `eval.jsonl`. File akan disalin ke folder `MyDrive/BYV-AI/data/`. Untuk sekarang, karena dataset lu belum ada, cell ini boleh dilewati.


In [6]:

from google.colab import files
uploaded = files.upload()  # pilih file .jsonl yang sudah disiapkan
for filename, content in uploaded.items():
    if not filename.lower().endswith(".jsonl"):
        print("Dilewati (bukan JSONL):", filename)
        continue
    destination = DATA_DIR / Path(filename).name
    destination.write_bytes(content)
    print("Tersimpan:", destination)


Saving eval.jsonl to eval.jsonl
Saving train.jsonl to train.jsonl
Tersimpan: /content/drive/MyDrive/BYV-AI/data/eval.jsonl
Tersimpan: /content/drive/MyDrive/BYV-AI/data/train.jsonl


## 6. Validasi dataset

Format yang diterima: setiap baris JSONL memiliki `messages`, berupa daftar pesan `system/user/assistant`. Jangan masukkan `tool_calls`, `function_call`, atau role `tool`: tool call BYV-AI adalah teks DSML di isi pesan assistant.

Untuk training sungguhan, buat `train.jsonl` dan (opsional) `eval.jsonl` di folder `MyDrive/BYV-AI/data/`. Idealnya siapkan **300–500 contoh awal yang ditinjau**, lalu tambah berdasarkan hasil evaluasi. Jangan salin private logs, credentials, API keys, atau data yang tidak punya izin untuk digunakan.

In [7]:
from collections import Counter
import re

VALID_ROLES = {"system", "user", "assistant"}
DSML_OPEN = "<｜DSML｜ calls>"
DSML_CLOSE = "</｜DSML｜ calls>"


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL error di {path}, baris {line_no}: {exc}")
    return rows


def validate_dsml_block(text, label="DSML"):
    if text.count(DSML_OPEN) != 1 or text.count(DSML_CLOSE) != 1:
        raise ValueError(f"{label}: wajib tepat satu outer calls block.")
    if not text.strip().startswith(DSML_OPEN) or not text.strip().endswith(DSML_CLOSE):
        raise ValueError(f"{label}: jangan letakkan prose di luar calls block.")
    invoke_open = re.findall(r'<｜DSML｜ invoke name="([^\"]+)">', text)
    invoke_close = text.count("</｜DSML｜ invoke>")
    if not invoke_open or len(invoke_open) != invoke_close:
        raise ValueError(f"{label}: jumlah invoke pembuka/penutup tidak cocok.")
    param_open = re.findall(r'<｜DSML｜ parameter name="([^\"]+)" string="(true|false)">', text)
    param_close = text.count("</｜DSML｜ parameter>")
    if len(param_open) != param_close:
        raise ValueError(f"{label}: jumlah parameter pembuka/penutup tidak cocok.")
    # Verify string=false payloads are valid JSON; string=true remains raw text by protocol.
    for match in re.finditer(r'<｜DSML｜ parameter name="([^\"]+)" string="(true|false)">(.*?)</｜DSML｜ parameter>', text, re.DOTALL):
        name, is_string, value = match.groups()
        if is_string == "false":
            try:
                json.loads(value)
            except json.JSONDecodeError as exc:
                raise ValueError(f"{label}: parameter non-string {name!r} bukan JSON valid: {exc}")
    return True


def validate_records(rows, label="dataset"):
    if not rows:
        raise ValueError(f"{label} kosong.")
    for idx, row in enumerate(rows, start=1):
        if not isinstance(row, dict) or not isinstance(row.get("messages"), list):
            raise ValueError(f"{label} contoh {idx}: wajib punya field messages berupa list.")
        if len(row["messages"]) < 2:
            raise ValueError(f"{label} contoh {idx}: minimal perlu user dan assistant.")
        for msg in row["messages"]:
            if msg.get("role") not in VALID_ROLES:
                raise ValueError(f"{label} contoh {idx}: role {msg.get('role')!r} tidak valid; gunakan system/user/assistant.")
            if not isinstance(msg.get("content"), str) or not msg["content"].strip():
                raise ValueError(f"{label} contoh {idx}: content harus string non-kosong.")
            if "tool_calls" in msg or "function_call" in msg:
                raise ValueError(f"{label} contoh {idx}: native tool-call metadata tidak diperbolehkan.")
        if row["messages"][-1]["role"] != "assistant":
            raise ValueError(f"{label} contoh {idx}: pesan terakhir harus assistant.")
        if "dsml" in row.get("category", "") or row.get("category") == "multi_turn_sequence":
            for assistant_idx, msg in enumerate(row["messages"], start=1):
                if msg["role"] == "assistant" and DSML_OPEN in msg["content"]:
                    validate_dsml_block(msg["content"], f"{label} contoh {idx}, assistant message {assistant_idx}")
    return True


if USE_STARTER_SMOKE_TEST:
    active_records = load_jsonl(SMOKE_PATH)
    validate_records(active_records, "starter smoke-test")
    print("MODE SMOKE TEST: dataset kecil starter dipakai hanya untuk menguji pipeline.")
else:
    if TRAIN_PATH.exists():
        active_records = load_jsonl(TRAIN_PATH)
        validate_records(active_records, "train")
        print("Train dataset ditemukan:", len(active_records), "contoh")
    else:
        active_records = []
        print("Dataset training belum ditemukan.")
        print("Isi file ini terlebih dahulu:", TRAIN_PATH)

if active_records:
    categories = Counter(row.get("category", "uncategorized") for row in active_records)
    print("Distribusi kategori:", dict(categories))
    dsml_samples = [row for row in active_records if "dsml" in row.get("category", "")]
    print("Contoh berkategori DSML:", len(dsml_samples))

Train dataset ditemukan: 280 contoh
Distribusi kategori: {'dsml_single_tool': 32, 'code_html_css': 24, 'code_javascript_typescript': 24, 'markdown': 24, 'reasoning': 24, 'code_python': 24, 'code_sql_other': 16, 'clarification_safety': 16, 'plain_text_with_tools': 24, 'plain_text': 24, 'dsml_multiple_independent': 16, 'multi_turn_sequence': 16, 'tool_result_to_markdown': 16}
Contoh berkategori DSML: 48


## 7. Siapkan train/eval split

Jika `eval.jsonl` tersedia, file itu digunakan sebagai validation set terpisah. Jika tidak tersedia, dataset training di-split 90/10 dengan seed tetap. Simpan data test final yang tidak pernah dipakai untuk training secara terpisah jika ingin melaporkan metrik yang lebih objektif.

In [8]:
from datasets import Dataset

TRAIN_DATASET = None
EVAL_DATASET = None
DATA_READY = False

if active_records:
    # Hanya field messages yang diberikan kepada SFTTrainer.
    train_source = [{"messages": row["messages"]} for row in active_records]
    if EVAL_PATH.exists() and not USE_STARTER_SMOKE_TEST:
        eval_records = load_jsonl(EVAL_PATH)
        validate_records(eval_records, "eval")
        eval_source = [{"messages": row["messages"]} for row in eval_records]
        TRAIN_DATASET = Dataset.from_list(train_source)
        EVAL_DATASET = Dataset.from_list(eval_source)
    else:
        if len(train_source) < 10:
            print("Dataset terlalu kecil untuk split 90/10; tambahkan contoh sebelum training.")
        else:
            split = Dataset.from_list(train_source).train_test_split(test_size=0.1, seed=SEED)
            TRAIN_DATASET, EVAL_DATASET = split["train"], split["test"]

    if TRAIN_DATASET is not None and EVAL_DATASET is not None:
        DATA_READY = True
        print("Train examples:", len(TRAIN_DATASET))
        print("Eval examples:", len(EVAL_DATASET))
        if len(active_records) < MIN_TRAIN_EXAMPLES and not ALLOW_SMALL_DATASET_FOR_SMOKE_TEST:
            print(f"Peringatan: dataset baru {len(active_records)} contoh. Untuk training sungguhan targetkan minimal {MIN_TRAIN_EXAMPLES}, ideal awal 300–500.")
else:
    print("DATA_READY=False. Notebook tetap dapat digunakan untuk cek GPU dan baseline.")

Train examples: 280
Eval examples: 35


## 8. Load tokenizer dan cek chat template Qwen3

Qwen3 menggunakan chat template khusus dan mendukung `enable_thinking=True`. Jangan membuat template chat sendiri atau menghapus special tokens model.

In [9]:
from transformers import AutoTokenizer

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

probe_messages = [
    {"role": "system", "content": BASE_SYSTEM},
    {"role": "user", "content": "Jelaskan apa itu BYV-AI dalam satu kalimat."}
]
probe_prompt = TOKENIZER.apply_chat_template(
    probe_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)
print(probe_prompt[:1200])
print("\nChat template check passed.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

<|im_start|>system
You are BYV-AI, a helpful reasoning language assistant.
Follow the user's instructions carefully. Think through tasks before answering, but do not expose private chain-of-thought. Provide useful concise explanations when appropriate.
Use plain text by default. Use Markdown when it improves readability. Generate complete source code when requested, preserving syntax and indentation. You can work with HTML, CSS, JavaScript, TypeScript, Python, SQL, and other programming languages.
Generate DSML only when an available tool is required and the current request needs it. DSML is ordinary assistant text, not native tool calling. Never invent tools, parameters, or execution results. Never claim an action succeeded before the gateway returns an actual result.
<|im_end|>
<|im_start|>user
Jelaskan apa itu BYV-AI dalam satu kalimat.<|im_end|>
<|im_start|>assistant


Chat template check passed.


## 9. Baseline evaluation sebelum training

Jalankan baseline ini setelah GPU tersedia. Empat prompt uji mewakili plain text, Markdown, source code, dan DSML. Output baseline disimpan agar dapat dibandingkan dengan adapter setelah training.

In [10]:
EVAL_CASES = [
    {
        "name": "plain_text",
        "messages": [
            {"role": "system", "content": BASE_SYSTEM},
            {"role": "user", "content": "Jelaskan secara singkat apa fungsi README dalam repository."}
        ]
    },
    {
        "name": "markdown",
        "messages": [
            {"role": "system", "content": BASE_SYSTEM},
            {"role": "user", "content": "Buat tabel Markdown yang membandingkan HTML dan CSS."}
        ]
    },
    {
        "name": "code_python",
        "messages": [
            {"role": "system", "content": BASE_SYSTEM},
            {"role": "user", "content": "Buat fungsi Python untuk menghitung faktorial dengan validasi input."}
        ]
    },
    {
        "name": "plain_text_with_tools",
        "messages": [
            {"role": "system", "content": BASE_SYSTEM + "\n\n" + DEMO_TOOL_PROMPT},
            {"role": "user", "content": "Jelaskan perbedaan HTML dan CSS. Jangan menjalankan tindakan apa pun."}
        ]
    },
    {
        "name": "dsml",
        "messages": [
            {"role": "system", "content": BASE_SYSTEM + "\n\n" + DEMO_TOOL_PROMPT},
            {"role": "user", "content": "Tampilkan direktori kerja saat ini."}
        ]
    }
]


def split_thinking_and_final(text):
    marker = "</think>"
    if marker in text:
        before, after = text.rsplit(marker, 1)
        thinking = before.replace("<think>", "").strip()
        final = after.strip()
    else:
        # Jangan salah menganggap thinking yang terpotong sebagai jawaban final.
        thinking = text.replace("<think>", "").strip() if "<think>" in text else ""
        final = ""

    # Hilangkan end-of-turn special tokens dari teks yang ditampilkan.
    for stop_token in ("<|im_end|>", "<|endoftext|>"):
        final = final.split(stop_token, 1)[0].strip()
    return thinking, final


def generate_text(model, tokenizer, messages, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=False)
    thinking, final = split_thinking_and_final(raw)
    return {"raw": raw, "thinking_present": bool(thinking), "final": final}


BASELINE_RESULTS = []
if GPU_AVAILABLE:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    baseline_bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True
    )
    baseline_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=baseline_bnb,
        device_map="auto",
        dtype=COMPUTE_DTYPE
    )
    baseline_model.eval()
    for case in EVAL_CASES:
        result = generate_text(baseline_model, TOKENIZER, case["messages"])
        BASELINE_RESULTS.append({"case": case["name"], **result})
        print("\n===== BASELINE:", case["name"], "=====")
        if result["final"]:
            print(result["final"][:3500])
        else:
            print("[Belum ada final answer; kemungkinan thinking mencapai max_new_tokens. Naikkan EVAL_MAX_NEW_TOKENS.]")
    del baseline_model
    torch.cuda.empty_cache()
else:
    print("Baseline GPU inference dilewati karena CUDA tidak tersedia.")

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


===== BASELINE: plain_text =====
**Fungsi README dalam repository**  
README adalah file panduan awal dalam repository, digunakan untuk memberikan informasi umum tentang proyek, seperti tujuan, fitur, instalasi, dan kontak. Ia membantu pengguna baru memahami apa yang dilakukan proyek dan bagaimana menggunakannya.

===== BASELINE: markdown =====
```markdown
| **Aspek**         | **HTML**                  | **CSS**                   |
|------------------|--------------------------|--------------------------|
| **Tujuan**        | Struktur konten         | Styling tampilan         |
| **Pengguna**      | HTML element (e.g. `<div>`) | CSS properties (e.g. `color`) |
| **Penggunaan**    | Tidak memiliki styling   | Diperlukan untuk styling  |
| **Contoh**        | `<p>Ini adalah paragraf</p>` | `color: red; font-size: 16px` |
| **Layout**        | Tidak mendefinisikan     | Menggunakan flexbox, grid, positioning |
| **Responsiveness** | Tidak (harus disetel) | Menggunakan media queries |
|

## 10. QLoRA training

Training hanya berjalan jika:

- CUDA tersedia;
- `train.jsonl` sudah disiapkan atau smoke-test diaktifkan secara eksplisit;
- ada train dan eval dataset;
- dataset cukup besar, kecuali flag smoke-test diubah secara sadar.

Preset untuk runtime saat ini (Tesla T4, CC 7.5, VRAM sekitar 14.56 GB): **FP16**, 4-bit NF4 + double quantization, LoRA rank 16, alpha 32, dropout 0.05, micro-batch 1, gradient accumulation 8, context 1024. Pada T4 jangan gunakan BF16; bila PyTorch melaporkan BF16 tersedia, itu dapat berasal dari pemeriksaan emulasi. Notebook memilih dtype berdasarkan compute capability dan akan mengaktifkan `fp16=True`, `bf16=False`. Jangan naikkan context/batch sebelum smoke test training berhasil dan peak VRAM diperiksa.

### Perbaikan dtype untuk Tesla T4

Notebook v0.1.2 memaksa model dasar dimuat dengan FP16 melalui `model_init_kwargs`, karena `fp16=True` pada TrainingArguments tidak otomatis mengubah dtype bobot yang diinfer dari `config.json`. Setelah trainer dibuat, seluruh parameter LoRA yang trainable dinormalisasi ke FP32 dan dtype-nya diperiksa sebelum training. Jika masih ada trainable BF16, notebook akan berhenti sebelum memanggil `trainer.train()`.

Jika sebelumnya training gagal dengan `_amp_foreach_non_finite_check_and_unscale_cuda ... BFloat16`, gunakan notebook revisi ini, restart runtime, lalu jalankan semua cell dari awal. Dataset yang sudah tersimpan di Google Drive tidak perlu di-upload ulang.


In [11]:
TRAINING_READY = True
training_blockers = []
if not GPU_AVAILABLE:
    TRAINING_READY = False
    training_blockers.append("GPU CUDA belum tersedia")
if not DATA_READY:
    TRAINING_READY = False
    training_blockers.append("train/eval dataset belum siap")
if active_records and len(active_records) < MIN_TRAIN_EXAMPLES and not (USE_STARTER_SMOKE_TEST and ALLOW_SMALL_DATASET_FOR_SMOKE_TEST):
    TRAINING_READY = False
    training_blockers.append(f"jumlah data {len(active_records)} masih di bawah batas {MIN_TRAIN_EXAMPLES}")

if TRAINING_READY:
    print("Semua prasyarat training terpenuhi.")
    print("GPU:", GPU_NAME, "| VRAM:", GPU_VRAM_GB, "GB")
    print("Train:", len(TRAIN_DATASET), "| Eval:", len(EVAL_DATASET))
else:
    print("Training belum dijalankan. Alasan:")
    for blocker in training_blockers:
        print("-", blocker)
    print("Notebook tetap bisa dipakai untuk menyiapkan Drive, dataset, dan baseline.")


Semua prasyarat training terpenuhi.
GPU: Tesla T4 | VRAM: 14.56 GB
Train: 280 | Eval: 35


In [12]:
if TRAINING_READY:
    import inspect
    from peft import LoraConfig
    from transformers import BitsAndBytesConfig
    from trl import SFTConfig, SFTTrainer

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True
    )

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ]
    )

    sft_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        run_name=RUN_NAME,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=2,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        optim="paged_adamw_8bit",
        logging_steps=5,
        logging_first_step=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=BF16_SUPPORTED,
        fp16=not BF16_SUPPORTED,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=MAX_LENGTH,
        packing=False,
        assistant_only_loss=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED
    )

    # Penting untuk Tesla T4:
    # Model card Qwen3 dapat menyatakan torch_dtype=bfloat16. Jangan biarkan
    # Transformers menginfer BF16 dari config; paksa seluruh bobot floating
    # base model dimuat dengan FP16. Transformers v5 memakai "dtype",
    # sedangkan Transformers v4 memakai "torch_dtype".
    import transformers
    transformers_major = int(transformers.__version__.split(".")[0])
    model_dtype_key = "dtype" if transformers_major >= 5 else "torch_dtype"
    sft_kwargs["model_init_kwargs"] = {model_dtype_key: COMPUTE_DTYPE}

    # Cek API TRL yang terpasang agar parameter penting tidak diam-diam dilewati.
    accepted = set(inspect.signature(SFTConfig).parameters)
    required_config = ["assistant_only_loss", "max_length", "eval_strategy", "model_init_kwargs"]
    missing = [name for name in required_config if name not in accepted]
    if missing:
        raise RuntimeError(
            f"Versi TRL saat ini tidak memiliki parameter penting {missing}. "
            "Periksa versi TRL dan dokumentasi SFTTrainer sebelum melanjutkan."
        )
    filtered_kwargs = {k: v for k, v in sft_kwargs.items() if k in accepted}
    ignored = sorted(set(sft_kwargs) - set(filtered_kwargs))
    if ignored:
        print("Parameter config yang tidak didukung versi TRL ini dan dilewati:", ignored)

    training_args = SFTConfig(**filtered_kwargs)
    trainer = SFTTrainer(
        model=MODEL_ID,
        args=training_args,
        train_dataset=TRAIN_DATASET,
        eval_dataset=EVAL_DATASET,
        processing_class=TOKENIZER,
        quantization_config=bnb_config,
        peft_config=lora_config
    )

    # Pastikan trainable LoRA parameters menggunakan FP32.
    # Ini menghindari GradScaler FP16 menerima gradient BF16 pada GPU T4.
    for parameter in trainer.model.parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            parameter.data = parameter.data.to(torch.float32)

    trainable_dtype_counts = Counter(
        str(parameter.dtype)
        for parameter in trainer.model.parameters()
        if parameter.requires_grad
    )
    print("Trainable parameter dtype:", dict(trainable_dtype_counts))

    if any(
        parameter.requires_grad and parameter.dtype == torch.bfloat16
        for parameter in trainer.model.parameters()
    ):
        raise RuntimeError(
            "Masih ada trainable BF16 parameter. Training dihentikan sebelum dimulai. "
            "Periksa model_init_kwargs dan pastikan base model dimuat dalam FP16."
        )

    print("Model QLoRA dan trainer siap. Base compute dtype:", COMPUTE_DTYPE)
else:
    print("Trainer tidak dibuat karena training belum siap.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Trainable parameter dtype: {'torch.float32': 392}
Model QLoRA dan trainer siap. Base compute dtype: torch.float16


In [13]:
if TRAINING_READY:
    train_result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR))  # Menyimpan adapter PEFT/LoRA
    TOKENIZER.save_pretrained(str(OUTPUT_DIR))

    metrics = train_result.metrics
    metrics["train_examples"] = len(TRAIN_DATASET)
    metrics["eval_examples"] = len(EVAL_DATASET)
    metrics["base_model"] = MODEL_ID
    metrics["gpu_name"] = GPU_NAME
    metrics["gpu_vram_gb"] = GPU_VRAM_GB
    metrics["max_length"] = MAX_LENGTH
    metrics["lora_r"] = LORA_R
    metrics["lora_alpha"] = LORA_ALPHA

    with open(EXPERIMENT_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2, default=str)

    with open(EXPERIMENT_DIR / "trainer_log_history.json", "w", encoding="utf-8") as f:
        json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2, default=str)

    print("Adapter tersimpan di:", OUTPUT_DIR)
    print(json.dumps(metrics, indent=2, ensure_ascii=False, default=str))
else:
    print("Training dilewati. Siapkan GPU dan dataset, lalu jalankan ulang cell mulai dari split dataset.")


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.911615,0.740587,0.721953,0.819415,131332.000000
2,0.616921,0.714249,0.614711,0.828855,262664.000000


Adapter tersimpan di: /content/drive/MyDrive/BYV-AI/outputs/byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2
{
  "train_runtime": 510.5487,
  "train_samples_per_second": 1.097,
  "train_steps_per_second": 0.137,
  "total_flos": 2248681145942016.0,
  "train_loss": 0.8290580136435373,
  "epoch": 2.0,
  "train_examples": 280,
  "eval_examples": 35,
  "base_model": "Qwen/Qwen3-1.7B",
  "gpu_name": "Tesla T4",
  "gpu_vram_gb": 14.56,
  "max_length": 1024,
  "lora_r": 16,
  "lora_alpha": 32
}


## 11. Evaluasi setelah training

Evaluasi di bawah bersifat inspeksi kualitatif: periksa apakah model menjawab plain text tanpa DSML, memakai Markdown/kode saat diminta, dan mengikuti nama tool/parameter yang disediakan. Jangan menganggap satu output yang tampak benar sebagai bukti kualitas umum.

In [14]:
if TRAINING_READY:
    POST_TRAIN_RESULTS = []
    trained_model = trainer.model
    trained_model.eval()

    for case in EVAL_CASES:
        result = generate_text(trained_model, TOKENIZER, case["messages"])
        POST_TRAIN_RESULTS.append({"case": case["name"], **result})
        print("\n===== AFTER TRAINING:", case["name"], "=====")
        print(result["final"][:3500])

    comparison_path = EXPERIMENT_DIR / "baseline_vs_finetuned.json"
    with open(comparison_path, "w", encoding="utf-8") as f:
        json.dump(
            {"baseline": BASELINE_RESULTS, "finetuned": POST_TRAIN_RESULTS},
            f, ensure_ascii=False, indent=2
        )
    print("Comparison saved:", comparison_path)
else:
    POST_TRAIN_RESULTS = []
    print("Evaluasi adapter dilewati karena training belum berjalan.")



===== AFTER TRAINING: plain_text =====
README adalah dokumen ringan yang menjelaskan tujuan, penggunaan, dan konfigurasi repository.

===== AFTER TRAINING: markdown =====
| Aspek | HTML | CSS |
|---|---|---|
| Tujuan | Membangun struktur dan konten | Membangun estetika dan responsif |
| Tidak menentukan styling | Menentukan properti visual |
| Membuat tampilan | Mengatur responsivitas dan layout |
| Dukungan struktur dan konten | Tidak menentukan struktur visual |

===== AFTER TRAINING: code_python =====
```python
def factorial(n: int, *, min_val: int = 0, max_val: int = 1000) -> int:
    if not (min_val <= n <= max_val):
        raise ValueError(f"Input harus antara {min_val} dan {max_val}")
    if n < 0:
        raise ValueError("Nilai tidak boleh negatif")
    if n == 0 or n == 1:
        return 1
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

===== AFTER TRAINING: plain_text_with_tools =====
HTML menentukan struktur dan konten halaman. CSS 

## 12. Simpan manifest eksperimen

Manifest membantu mengulang eksperimen. Jangan memasukkan token Hugging Face, API key, credential, atau data pribadi ke manifest maupun repo.

In [15]:
import importlib.metadata as metadata

package_names = ["torch", "transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes"]
package_versions = {}
for package in package_names:
    try:
        package_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        package_versions[package] = None

manifest = {
    "run_name": RUN_NAME,
    "base_model": MODEL_ID,
    "method": "SFT + QLoRA",
    "quantization": "4-bit NF4 with double quantization",
    "thinking": "Qwen3 chat template with enable_thinking=True at inference",
    "native_tool_calling": False,
    "dsml_representation": "assistant text content",
    "dataset_mode": "starter_smoke_test" if USE_STARTER_SMOKE_TEST else "user_dataset",
    "training_ready": TRAINING_READY,
    "training_started": bool(TRAINING_READY and "trainer" in globals() and trainer.state.global_step > 0),
    "dataset_count": len(active_records),
    "gpu_name": GPU_NAME,
    "gpu_vram_gb": GPU_VRAM_GB,
    "gpu_compute_capability": GPU_COMPUTE_CAPABILITY,
    "compute_dtype": str(COMPUTE_DTYPE),
    "bf16_native_supported": BF16_SUPPORTED,
    "bf16_emulation_reported_by_pytorch": BF16_EMULATION_REPORTED,
    "max_length": MAX_LENGTH,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "packages": package_versions
}

with open(EXPERIMENT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Manifest:", EXPERIMENT_DIR / "manifest.json")
print(json.dumps(manifest, ensure_ascii=False, indent=2))

Manifest: /content/drive/MyDrive/BYV-AI/experiments/byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2/manifest.json
{
  "run_name": "byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2",
  "base_model": "Qwen/Qwen3-1.7B",
  "method": "SFT + QLoRA",
  "quantization": "4-bit NF4 with double quantization",
  "thinking": "Qwen3 chat template with enable_thinking=True at inference",
  "native_tool_calling": false,
  "dsml_representation": "assistant text content",
  "dataset_mode": "user_dataset",
  "training_ready": true,
  "training_started": true,
  "dataset_count": 280,
  "gpu_name": "Tesla T4",
  "gpu_vram_gb": 14.56,
  "gpu_compute_capability": "7.5",
  "compute_dtype": "torch.float16",
  "bf16_native_supported": false,
  "bf16_emulation_reported_by_pytorch": true,
  "max_length": 1024,
  "epochs": 2,
  "learning_rate": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "packages": {
    "torch": "2.11.0+cu130",
    "transformers": "5.22.1",
    "datasets": "5.1.0",
    "accelerate": "1.15

## 13. Hasil dan langkah selanjutnya

Jika training berhasil, folder output berisi adapter LoRA dan tokenizer. **Ini belum GGUF dan belum model mandiri hasil merge.** Tahap export/merge/quantization dilakukan terpisah setelah adapter dievaluasi.

Checklist sebelum menyebut BYV-AI v0.1 berhasil:

- [ ] Plain text tidak menghasilkan DSML tanpa alasan.
- [ ] DSML hanya menggunakan tool yang memang disediakan di prompt.
- [ ] Nama tool dan parameter sesuai schema.
- [ ] Satu calls block digunakan untuk satu respons permintaan tool.
- [ ] Tool result nyata dapat diringkas menjadi Markdown.
- [ ] HTML/CSS/JavaScript/Python yang diminta menghasilkan source code yang masuk akal.
- [ ] Model tidak mengklaim kode sudah dijalankan jika belum ada hasil eksekusi.
- [ ] Bandingkan baseline dengan fine-tuned pada test set yang tidak dipakai training.
- [ ] Catat RAM/VRAM, latency, token/s, dan kualitas setelah export ke CPU.

**Peringatan dataset:** contoh starter di notebook hanya untuk memeriksa format dan smoke-test. Untuk model yang benar-benar berguna, kurasi data yang jauh lebih banyak, variasikan prompt, uji kasus ambigu/negatif, periksa lisensi, dan simpan test set terpisah. Fine-tuning bukan jaminan selalu valid; gateway tetap wajib mem-parse, memvalidasi schema, memeriksa izin, dan mengamankan eksekusi DSML.


## Referensi teknis

- [Qwen3-1.7B model card](https://huggingface.co/Qwen/Qwen3-1.7B) — model, chat template, `enable_thinking=True`, dan sampling yang direkomendasikan.
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) — conversational datasets dan `assistant_only_loss=True`.
- [TRL PEFT / QLoRA integration](https://huggingface.co/docs/trl/peft_integration) — `quantization_config` 4-bit NF4 bersama `peft_config`.
- [Transformers bitsandbytes integration](https://huggingface.co/docs/transformers/quantization/bitsandbytes) — NF4 dan hardware compatibility.
- [Google Colab FAQ](https://research.google.com/colaboratory/intl/id/faq.html) — resource GPU Colab bersifat dinamis dan tidak dijamin.

- [NVIDIA CUDA GPU Compute Capability](https://developer.nvidia.com/cuda/gpus) dan [CUDA Compute Capabilities — Tensor Core data types](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/compute-capabilities.html) — T4 berada di CC 7.5; Tensor Core input types mencakup FP16, bukan native BF16.
- [PyTorch `torch.cuda.is_bf16_supported`](https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_bf16_supported.html) — default `including_emulation=True`, sehingga nilai True tidak selalu berarti native BF16 Tensor Core support.
